# Dark Matter Data Handling with Gammapy

In any dark matter indirect detection analysis, the first step is to define
what data you are going to work with. Gammapy supports two complementary 
approaches:

- **Simulated data:** you generate synthetic observations from a theoretical 
  model. This is useful for computing expected sensitivities, deriving expected
  upper limits, and validating your analysis pipeline before applying it to 
  real observations.

- **Real data:** you load actual telescope observations in DL3 format and 
  reduce them to a dataset ready for statistical analysis.

In both cases, the downstream analysis — likelihood fitting, upper limit 
computation, combination of targets — is identical in Gammapy. This means 
that a pipeline developed on simulations can be directly applied to real data 
with minimal changes.

## Simulating Dark Matter Observations

To simulate an observation we need three ingredients:

- **A signal model:** the expected gamma-ray spectrum from DM annihilation 
  or decay, built from the J/D-factor and the particle physics spectrum 
  (see Tutorial **XXXXX**).
- **A background model:** the residual cosmic-ray background surviving 
  the gamma/hadron separation cuts.
- **Instrument Response Functions (IRFs):** the telescope's effective area, 
  energy dispersion, and PSF, which translate the physical flux into 
  observable counts.

We will simulate observations of the **Draco dwarf spheroidal galaxy** using 
CTA-North IRFs (prod5 configuration), and the `DarkMatterAnnihilationSpectralModel` 
from Tutorial 1 as our signal model.

Two simulation strategies are available in Gammapy: Asimov and Monte Carlo.

In the next steops we are going to cover all the necessary components and the final simulation with both approaches.



### Setup

In [ ]:
from gammapy.data import Observation
from gammapy.datasets import MapDataset
from gammapy.irf import load_irf_dict_from_file
from gammapy.makers import (
    MapDatasetMaker,
    SafeMaskMaker
)
from gammapy.maps import MapAxis, WcsGeom, WcsNDMap
from gammapy.modeling.models import (
    FoVBackgroundModel,
    Models,
    SkyModel,
    PointSpatialModel
)
from gammapy.astro.darkmatter import (
    DarkMatterAnnihilationSpectralModel
)
from gammapy.astro.darkmatter.profiles import EinastoProfile

import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import numpy as np
import matplotlib.pyplot as plt



### Target source

We define our target source. For a realistic simulation, we need to know the object's elevation in the sky from the telescope's location at a specific time. This affects which IRF (zenith angle dependent) we should use.

In [ ]:
# Source coordinates
draco_pos = SkyCoord(ra=260.05167 * u.deg, dec=57.915 * u.deg, frame="icrs")
draco_dist = 76 * u.kpc

# Location coordinates, i.e CTA-North
cta_norte = EarthLocation(lat=28.762 * u.deg, lon=-17.89 * u.deg, height=2200 * u.m)  

# Observation start time
obs_time = Time("2025-06-17 01:00:00")  # UTC

altaz = AltAz(obstime=obs_time, location=cta_norte)
draco_altaz = draco_pos.transform_to(altaz)

# Altitude and zenith angle
altitude = draco_altaz.alt.deg
zenith_angle = 90 - altitude

print(f"Altitude: {altitude:.2f}°")
print(f"Zenith angle: {zenith_angle:.2f}°")

### Dark Matter model parameters
We define the particle physics properties: the annihilation channel (e.g., h bosons), the particle mass, and the J-Factor (the line-of-sight integral of the DM density squared).

In [ ]:
# DM parameters: mass and channel
channel = "h"
massDM = 10 * u.TeV

# Density profile
rho_s_msun_kpc3 = 1.3e7 * (u.M_sun / u.kpc**3)  
# From M_sun/kpc³ to GeV/cm³
rho_s_GeV_cm3 = rho_s_msun_kpc3.to(u.GeV / u.cm**3, equivalencies=u.mass_energy())

# Jfactor
jfact_draco = 1.66e23 * u.GeV**2 / u.cm**5 #Multiplied by a number from which the signal will be detected, in this case is 5

draco_profile = EinastoProfile(
    r_s=0.91 * u.kpc,       
    rho_s=rho_s_GeV_cm3
)

### Map geometry and energy range
We define the energy range for the simulation and the spatial resolution of our World Coordinate System (WCS) map.

In [ ]:
# Set energy bounds
energy_edges = np.logspace(-1, 2, 15) 
energy_reco = MapAxis.from_edges(energy_edges, unit="TeV", name="energy", interp="log")
energy_true = MapAxis.from_edges(energy_edges, unit="TeV", name="energy_true", interp="log")

# Geometry map
geom_draco = WcsGeom.create(
    skydir=draco_pos,
    binsz=0.01,
    width=5.0,
    frame="icrs",
    axes=[energy_reco]  
)

### Sky model
We combine the spatial model (treating Draco as a point source) with the DM annihilation spectral model. 

The Spatial Model defines the shape and position of our target. We use a PointSpatialModel centered on Draco. While DM halos are technically extended, Draco is compact enough that a point-source approximation is a standard starting point relative to the telescope's resolution.

The Spectral Model defines the gamma-ray flux as a function of energy. We use the DarkMatterAnnihilationSpectralModel, which combines Particle Physics (mass and annihilation channel) with Astrophysics (the J-Factor).

In [ ]:
# Spatial model
spatial_model = PointSpatialModel(
    lon_0=draco_pos.ra,
    lat_0=draco_pos.dec,
    frame="icrs"
)

# Spectral model
spectral_model = DarkMatterAnnihilationSpectralModel(
    mass=massDM, channel=channel, jfactor=jfact_draco)

# Combined model
model_simu = SkyModel(
    spatial_model=spatial_model,
    spectral_model=spectral_model,
    name="draco-dm"
)

### IRF and observation setup
We load the IRFs (Instrument Response Functions), which represent the "fingerprint" of the telescope. They describe how the instrument distorts the signal: Effective Area (efficiency), PSF (spatial blurring), and Energy Dispersion (energy resolution). In this simulation, they translate our theoretical Dark Matter flux into the realistic "counts" the CTA would actually detect.

In [ ]:
# Load CTA Prod5 IRFs
irf_path = r"/Users/alexcervino/Desktop/DARKMATTER/ksp_dsph_material/IRFs/cta/prod5-v0.1/bcf/North_z20_N_50h/Prod5-North-20deg-NorthAz-4LSTs09MSTs.180000s-v0.1.fits.gz"
irfs = load_irf_dict_from_file(irf_path)

# Create the Observation: 500 hours of livetime
livetime = 500 * u.h
obs = Observation.create(
    pointing=draco_pos, 
    livetime=livetime, 
    irfs=irfs, 
    reference_time=obs_time
)

### Simulation
In this final step, we apply the models and IRFs to generate a synthetic event count map. In this case we are going to simulate using the two approaches mentioned before, using Asimov and Monte Carlo.

In [ ]:
# 1. Create an empty MapDataset
empty = MapDataset.create(geom=geom_draco, name="dataset-simu-draco", energy_axis_true=energy_true)

# 2. Setup Maker to calculate exposure, background, PSF, and energy dispersion
maker = MapDatasetMaker(selection=["exposure", "background", "psf", "edisp"])
maker_safe_mask = SafeMaskMaker(methods=["offset-max"], offset_max=2.5 * u.deg)

# 3. Run the maker and attach models
dataset = maker.run(empty, obs)
dataset = maker_safe_mask.run(dataset, obs)

# Attach the DM model and a Field-of-View background model
bkg_model = FoVBackgroundModel(dataset_name="dataset-simu-draco")
dataset.models = Models([model_simu, bkg_model])

### Asimov dataset
The Asimov dataset is a mathematically convenient dataset where the counts are set equal exactly to the model prediction, with no statistical noise. This concept (Cowan et al. 2011) is widely used in HEP and gamma-ray astronomy because a single Asimov dataset directly yields the *median expected* sensitivity, without needing to average over hundreds of MC realizations.

 It corresponds to the "median experiment" — what you would observe on average over many 
realizations. It is the standard approach for computing **expected upper 
limits** and **sensitivity curves**, and is much faster than running 
many Monte Carlo realizations.

In [ ]:
# Copy the base dataset (models already attached, counts not yet filled)
dataset_asimov = dataset.copy(name="dataset-asimov-draco")

# Set counts equal to the model prediction — no Poisson sampling
dataset_asimov.counts = dataset_asimov.npred()

print("=== Asimov Dataset ===")
print(f"Predicted counts : {dataset_asimov.npred().data.sum():.2f}")
print(f"Asimov counts    : {dataset_asimov.counts.data.sum():.2f}")


#### Monte Carlo dataset
A Monte Carlo (MC) dataset is generated by drawing Poisson random numbers 
around the expected counts. Each realization represents one possible 
outcome of the actual observation. Running many MC realizations allows 
you to **validate** the statistical coverage of your analysis and to 
cross-check the sensitivity bands derived from the Asimov approach.

In [ ]:
# Copy the base dataset
dataset_mc = dataset.copy(name="dataset-mc-draco")

# Sample Poisson fluctuations around the prediction
# random_state fixes the seed for reproducibility
dataset_mc.fake(random_state=42)

print("=== MC Observation ===")
print(f"Predicted counts : {dataset_mc.npred().data.sum():.2f}")
print(f"Simulated counts : {dataset_mc.counts.data.sum()}")

# Simulation inspection

In [ ]:
# ─────────────────────────────────────────────
# Interactive comparison: Asimov vs MC
# ─────────────────────────────────────────────
print("=== Asimov ===")
dataset_asimov.counts.smooth(0.05 * u.deg).plot_interactive(add_cbar=True, stretch="linear")
plt.show()

print("=== MC ===")
dataset_mc.counts.smooth(0.05 * u.deg).plot_interactive(add_cbar=True, stretch="linear")
plt.show()

# ─────────────────────────────────────────────
# Static comparison: summed over all energies
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

dataset_asimov.counts.reduce_over_axes().smooth(0.05 * u.deg).plot(
    ax=axes[0], add_cbar=True
)
axes[0].set_title("Asimov (no fluctuations)")

dataset_mc.counts.reduce_over_axes().smooth(0.05 * u.deg).plot(
    ax=axes[1], add_cbar=True
)
axes[1].set_title("MC Observation (seed=42)")

plt.suptitle(f"Count maps — DM annihilation ({channel} channel, {massDM})")
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# Static comparison: restricted energy range
# ─────────────────────────────────────────────
energy_min = 3.16 * u.TeV
energy_max = 5.18 * u.TeV
energy_axis = dataset_asimov.counts.geom.axes["energy"]
energy_indices = np.where(
    (energy_axis.center >= energy_min) & (energy_axis.center <= energy_max)
)[0]

geom_image = dataset_asimov.counts.geom.to_image()

for ds, label in [(dataset_asimov, "Asimov"), (dataset_mc, "MC Observation")]:
    counts_sum = ds.counts.data[energy_indices, :, :].sum(axis=0)
    counts_map = WcsNDMap(geom=geom_image, data=counts_sum)
    smoothed_map = counts_map.smooth(0.05 * u.deg)

    fig, ax = plt.subplots()
    smoothed_map.plot(ax=ax, add_cbar=True)
    ax.set_title(f"{label} — {energy_min:.2f} to {energy_max:.2f}")
    plt.show()

In [ ]:
# General check of the dataset
dataset_asimov.peek()

### Statistical validation: The background-only hypothesis
To confirm that any detected signal is truly from Dark Matter and not just a statistical fluke of the background, we perform a Null Hypothesis test.

We create a deepcopy of our simulation. This "frozen" copy contains the exact same simulated events (counts) but only knows about the Background model, ignoring the Dark Matter component.

Excess & Significance Estimation: We use the ExcessMapEstimator to correlate the data. The excess map shows the "leftover" counts after subtracting the background model. Since this dataset contains DM events but the model expects only background, we should see a clear excess at Draco's position. The significance map (Li & Ma) converts that excess into standard deviations (σ). This tells us if the "bump" we see is statistically significant or just noise.

In [ ]:
#Duplicate the dataset just to store the background model
from copy import deepcopy

dataset_only_bkg = deepcopy(dataset)

# Create background model
bkg_model_only = FoVBackgroundModel(dataset_name=dataset_only_bkg.name)
dataset_only_bkg.models = Models([bkg_model_only])


In [ ]:
print(dataset_only_bkg)

In [ ]:
# Excess map and significance
from gammapy.estimators import ExcessMapEstimator

estimator = ExcessMapEstimator(
    0.1 * u.deg, selection_optional=[], energy_edges=[0.4, 1] * u.TeV
)

maps = estimator.run(dataset_only_bkg)

plt.figure(figsize=(10, 10))
ax1 = plt.subplot(121, projection=maps.sqrt_ts.geom.wcs)
ax2 = plt.subplot(122, projection=maps.sqrt_ts.geom.wcs)

ax1.set_title("Significance map")
maps.sqrt_ts.plot(ax=ax1, add_cbar=True)

ax2.set_title("Excess map")
maps.npred_excess.plot(ax=ax2, add_cbar=True)
plt.show()


## References

- **Source position and distance of Draco:** McConnachie, A. W. 2012, *AJ*, 144, 4
  — The observed properties of dwarf galaxies in and around the Local Group.
  [arXiv:1204.1562](https://arxiv.org/abs/1204.1562)

- **DM density profile and J-factor (Draco):** Bonnivard, V. et al. 2015,
  *MNRAS*, 453, 849
  — Dark matter annihilation and decay in dwarf spheroidal galaxies: The classical and ultrafaint dSphs.
  [arXiv:1504.02048](https://arxiv.org/abs/1504.02048)